## Step 1. Import Modules

In [ ]:
import pandas as pd
import json
import os

## Step 2. Specify the file paths

In [ ]:
csv_file_path = '../csv2JSON/OpenIndexMaps_Aardvark_workshop.csv'  # the name of your CSV
reference_uris_file_path = '../aardvark-profile/referenceURIs.csv'  # CSV mapping reference URIs and labels
full_schema_file_path = '../aardvark-profile/aardvark.csv'  # CSV mapping OGM Aardvark fields and labels
output_dir = 'json_output'  # Output directory

## Step 3. Define the function

In [ ]:
def convert_csv_to_json(csv_file_path, reference_uris_file_path, full_schema_file_path, output_dir):
    # Load the CSV data
    csv_data = pd.read_csv(csv_file_path)
    # Load the reference URIs data
    reference_uris_data = pd.read_csv(reference_uris_file_path)
    reference_uri_dict = dict(zip(reference_uris_data['LABEL'], reference_uris_data['URI']))
    # Load the full schema data
    full_schema_data = pd.read_csv(full_schema_file_path)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Function to handle multivalued fields
    def split_multivalues(val, format):
        value_list = val.split('|') if pd.notna(val) and '|' in val else [val]
        if format == "im":
            value_list = [int(item) for item in value_list]
        return value_list
    
    def normalize_ark_identifier(value):
        if pd.isna(value):
            return value
        identifier = str(value).strip()
        if identifier.startswith('ark:-'):
            return identifier.replace('ark:-', 'ark:/', 1).replace('-', '/', 1)
        return identifier
    
    # Function to construct JSON data from a row
    def construct_json_data(row):
        json_data = {}
        for _, schema_row in full_schema_data.iterrows():
            label = schema_row['Label']
            field_name = schema_row['Field Name']
            field_type = schema_row['Field Type']
            
            if field_name in ["dct_references_s"]:  # Handle references separately
                references = {}
                for ref_label in reference_uri_dict.keys():
                    if pd.notna(row.get(ref_label)):
                        references[reference_uri_dict[ref_label]] = row[ref_label]
                if str(row.get("Format", "")).strip() == "GeoJSON" and pd.notna(row.get("Download")):
                    references[reference_uri_dict["GeoJSON"]] = row["Download"]
                if references:
                    json_data[field_name] = json.dumps(references)
            elif pd.notna(row.get(label)) or (label == "Identifier" and pd.notna(row.get("ID"))):
                source_value = row.get("ID") if label == "Identifier" and pd.notna(row.get("ID")) else row.get(label)
                if label == "Identifier":
                    source_value = normalize_ark_identifier(source_value)
                if field_type == "Array":
                    json_data[field_name] = split_multivalues(str(source_value), field_name[-2:])
                elif field_type == "Boolean or string":
                    json_data[field_name] = bool(row.get(label, "false"))
                else:
                    data = source_value
                    if isinstance(data, float):
                        data = int(data)
                    json_data[field_name] = str(data)
        
        # json_data["gbl_mdVersion_s"] = "Aardvark" # We don't need this since we define it in the CSV
        return json_data

    # Iterate over each row in the CSV and generate JSON files
    for index, row in csv_data.iterrows():
        json_data = construct_json_data(row)
        
        # Match the existing edu.uwm metadata-aardvark filename convention
        id_part = str(row.get('ID', 'DEFAULT')).strip()
        filename_id = id_part.removeprefix('ark:-77981-')
        if filename_id == id_part:
            filename_id = id_part.split('/')[-1].split('-')[-1]
        file_name = f"{filename_id}_BL_Aardvark.json"
        file_path = os.path.join(output_dir, file_name)
        
        # Write the JSON data to a file
        with open(file_path, 'w', encoding="utf-8") as json_file:
            json.dump(json_data, json_file, indent=4, ensure_ascii=False)

## Step 4: Run the script

In [ ]:
# Convert the CSV to individual JSON files
convert_csv_to_json(csv_file_path, reference_uris_file_path, full_schema_file_path, output_dir)

# Print a message indicating completion
print(f'JSON files generated in directory: {output_dir}')